# 🤖 AI-Powered Enterprise RAG Architecture

## Interactive RFP → Proposal Approach Development

**Features:**
- 🧠 RAG-based learning from your RFP/Proposal repository
- 🔍 Full transparency into AI reasoning and evidence
- 🛠️ Human-in-the-loop refinement at every stage
- 📊 Professional visualizations and exports
- 💾 All data stays local on your machine

---

### How to Use:
1. **Run all cells in sequence** (Cell → Run All)
2. **Follow the workflow** through each section
3. **Review and refine** at each checkpoint
4. **Export** your professional deliverables

### Sections:
1. ⚙️ Setup & Configuration
2. 📚 Repository Management (Index your RFPs and Proposals)
3. 🎯 RFP Analysis (Extract requirements and learn from repository)
4. 🏗️ High-Level Structure (Build workstreams with dependencies)
5. 📋 Detailed Breakdown (Modules and activities)
6. 📊 Visualizations (Professional timelines and charts)
7. 📤 Export (Excel, PowerPoint, PDF)

---

## 1️⃣ Setup & Configuration

### Prerequisites:
- Python 3.10+
- OpenAI API key
- Your confidential RFPs and Proposals in `data/` folder

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Import our modules
from data_models import ProjectStructure, Workstream, Module
from rag_engine import RAGEngine
from agents import WorkstreamAgent, DependencyAgent, ModuleAgent, TimelineOptimizer
from utils import (
    validate_environment,
    save_project,
    load_project,
    print_project_summary,
    calculate_project_metrics,
    format_reasoning_for_display
)

print("✅ All modules imported successfully!")

In [ ]:
# Load environment variables
env_path = Path.cwd().parent / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print("✅ Loaded .env file")
else:
    print("⚠️ No .env file found. Please create one from .env.example")

# Validate environment
validation = validate_environment()

if validation['valid']:
    print("\n✅ Environment validation passed!")
else:
    print("\n❌ Environment validation failed:")
    for error in validation['errors']:
        print(f"   • {error}")

if validation['warnings']:
    print("\n⚠️ Warnings:")
    for warning in validation['warnings']:
        print(f"   • {warning}")

# Get API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4')

if not OPENAI_API_KEY:
    print("\n⚠️ Please set your OPENAI_API_KEY in the .env file")
else:
    print(f"\n✅ Using model: {OPENAI_MODEL}")
    print(f"   API key: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}")

---

## 2️⃣ Repository Management

### Add your confidential RFPs and Proposals:

1. Place your **RFP PDFs** in: `data/rfps/`
2. Place your **Proposal PPTX files** in: `data/proposals/`
3. Run the cell below to index them

**Note:** These files will NOT be committed to GitHub (protected by .gitignore)

In [ ]:
# Initialize RAG Engine
print("🔧 Initializing RAG Engine...\n")

rag_engine = RAGEngine(
    api_key=OPENAI_API_KEY,
    db_path="../data/chroma_db",
    verbose=True
)

print("\n✅ RAG Engine initialized!")

In [ ]:
# Check current repository status
stats = rag_engine.get_collection_stats()

print("\n📊 Current Repository Status:")
print(f"   Total chunks indexed: {stats['total_chunks']}")
print(f"   Unique source files: {stats['unique_sources']}")
print(f"   RFPs: {stats['doc_types'].get('rfp', 0)} chunks")
print(f"   Proposals: {stats['doc_types'].get('proposal', 0)} chunks")

if stats['sources']:
    print("\n📄 Source files:")
    for source, count in sorted(stats['sources'].items(), key=lambda x: -x[1])[:10]:
        print(f"   • {source}: {count} chunks")

In [ ]:
# Index documents (run this to refresh the index)
print("🔄 Starting document indexing...\n")

indexing_result = rag_engine.index_documents(
    rfp_dir="../data/rfps",
    proposal_dir="../data/proposals"
)

print("\n" + "="*70)
print("INDEXING COMPLETE")
print("="*70)
print(f"Total chunks: {indexing_result['total_chunks']}")
print(f"RFP files: {indexing_result['rfp_count']}")
print(f"Proposal files: {indexing_result['proposal_count']}")
print(f"Duration: {indexing_result['duration_seconds']:.2f}s")

if indexing_result['errors']:
    print(f"\n⚠️ Errors: {len(indexing_result['errors'])}")
    for error in indexing_result['errors'][:5]:
        print(f"   • {error}")

---

## 3️⃣ RFP Analysis

### Load your RFP or paste the text

In [ ]:
# Option 1: Load from file
rfp_files = list(Path("../data/rfps").glob("*.pdf"))

if rfp_files:
    print("📄 Available RFP files:")
    for i, f in enumerate(rfp_files, 1):
        print(f"   {i}. {f.name}")
    
    # Select RFP file (change index as needed)
    selected_rfp_index = 0  # First file (change this!)
    
    if selected_rfp_index < len(rfp_files):
        selected_rfp = rfp_files[selected_rfp_index]
        print(f"\n✅ Selected: {selected_rfp.name}")
        print(f"   Extracting text...")
        
        rfp_text = rag_engine.extract_pdf_text(selected_rfp)
        
        print(f"   Extracted {len(rfp_text):,} characters")
        print(f"   ~{len(rfp_text.split()):,} words")
    else:
        print("\n⚠️ Invalid selection. Please adjust selected_rfp_index")
        rfp_text = ""
else:
    print("⚠️ No RFP files found. Please add PDFs to data/rfps/")
    rfp_text = ""

# Option 2: Or paste RFP text directly
# rfp_text = """Paste your RFP text here..."""

In [ ]:
# Preview RFP content
if rfp_text:
    print("📄 RFP Content Preview (first 1000 characters):\n")
    print("="*70)
    print(rfp_text[:1000])
    print("...")
    print("="*70)
else:
    print("⚠️ No RFP content loaded")

### Initialize Project Structure

In [ ]:
# Create new project
project_name = input("Enter project name (or press Enter for default): ").strip()
if not project_name:
    project_name = f"Project_{Path(selected_rfp).stem if 'selected_rfp' in locals() else 'Untitled'}"

project = ProjectStructure(
    project_name=project_name,
    rfp_text=rfp_text
)

print(f"\n✅ Created project: {project.project_name}")

---

## 4️⃣ High-Level Structure: Workstream Extraction

### Step 1: Extract Explicit Workstreams from RFP

In [ ]:
# Initialize Workstream Agent
workstream_agent = WorkstreamAgent(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    verbose=True
)

print("✅ Workstream Agent initialized")

In [ ]:
# Extract explicit workstreams
if rfp_text:
    explicit_workstreams, explicit_duration, reasoning = workstream_agent.extract_explicit_workstreams(rfp_text)
    
    # Save reasoning
    project.add_agent_reasoning(reasoning)
    
    # Display results
    print("\n" + "="*70)
    print("EXTRACTION RESULTS")
    print("="*70)
    
    if explicit_workstreams:
        print(f"\n✅ Found {len(explicit_workstreams)} explicit workstreams:")
        for i, ws in enumerate(explicit_workstreams, 1):
            print(f"   {i}. {ws}")
        
        if explicit_duration:
            print(f"\n⏱️  Explicit duration: {explicit_duration} weeks")
            project.total_duration_weeks = explicit_duration
    else:
        print("\nℹ️ No explicit workstreams found in RFP")
        print("   Will learn from repository in next step")
else:
    print("⚠️ No RFP text available for extraction")

### Step 2: Learn from Repository (if needed)

In [ ]:
# Learn workstreams from repository if not explicitly mentioned
if not explicit_workstreams or len(explicit_workstreams) < 3:
    print("\n🧠 Learning workstream structure from repository...\n")
    
    learned_workstreams, reasoning = workstream_agent.learn_from_repository(
        rfp_text=rfp_text,
        rag_engine=rag_engine
    )
    
    # Save reasoning
    project.add_agent_reasoning(reasoning)
    
    # Use learned workstreams
    project.workstreams = learned_workstreams
    
    print(f"\n✅ Generated {len(learned_workstreams)} workstreams from repository")
    
else:
    # Convert explicit workstreams to Workstream objects
    print("\n✅ Using explicit workstreams from RFP")
    
    for ws_name in explicit_workstreams:
        workstream = Workstream(
            name=ws_name,
            duration_weeks=4,  # Default, will be refined
            rationale="Explicitly mentioned in RFP",
            confidence_score=0.9
        )
        project.workstreams.append(workstream)

# Display workstreams
print("\n" + "="*70)
print("CURRENT WORKSTREAMS")
print("="*70)

for i, ws in enumerate(project.workstreams, 1):
    print(f"\n{i}. {ws.name}")
    print(f"   Duration: {ws.duration_weeks} weeks")
    print(f"   Confidence: {ws.confidence_score:.2f}")
    print(f"   Rationale: {ws.rationale[:100]}...")

total_weeks = sum(ws.duration_weeks for ws in project.workstreams)
print(f"\n📊 Total Duration: {total_weeks} weeks (~{total_weeks/4:.1f} months)")

### 🛠️ Human-in-the-Loop: Refine Workstreams

**Review and edit the workstreams above before proceeding.**

In [ ]:
# Interactive workstream refinement
print("\n🛠️ WORKSTREAM REFINEMENT")
print("="*70)
print("\nOptions:")
print("  1. Edit workstream name")
print("  2. Edit workstream duration")
print("  3. Remove workstream")
print("  4. Add new workstream")
print("  5. Continue to next step")

while True:
    choice = input("\nEnter choice (1-5): ").strip()
    
    if choice == "5":
        print("✅ Workstreams locked. Proceeding to dependencies...")
        break
    
    elif choice == "1":
        # Edit name
        for i, ws in enumerate(project.workstreams, 1):
            print(f"   {i}. {ws.name}")
        
        idx = int(input("Select workstream number: ")) - 1
        if 0 <= idx < len(project.workstreams):
            new_name = input("Enter new name: ").strip()
            if new_name:
                old_name = project.workstreams[idx].name
                project.workstreams[idx].name = new_name
                print(f"✅ Renamed '{old_name}' to '{new_name}'")
    
    elif choice == "2":
        # Edit duration
        for i, ws in enumerate(project.workstreams, 1):
            print(f"   {i}. {ws.name} ({ws.duration_weeks}w)")
        
        idx = int(input("Select workstream number: ")) - 1
        if 0 <= idx < len(project.workstreams):
            new_duration = int(input(f"Enter new duration (current: {project.workstreams[idx].duration_weeks}w): "))
            project.workstreams[idx].duration_weeks = new_duration
            print(f"✅ Updated duration to {new_duration} weeks")
    
    elif choice == "3":
        # Remove
        for i, ws in enumerate(project.workstreams, 1):
            print(f"   {i}. {ws.name}")
        
        idx = int(input("Select workstream to remove: ")) - 1
        if 0 <= idx < len(project.workstreams):
            removed = project.workstreams.pop(idx)
            print(f"✅ Removed '{removed.name}'")
    
    elif choice == "4":
        # Add new
        name = input("Enter workstream name: ").strip()
        duration = int(input("Enter duration (weeks): "))
        rationale = input("Enter rationale: ").strip()
        
        new_ws = Workstream(
            name=name,
            duration_weeks=duration,
            rationale=rationale or "Added manually"
        )
        project.workstreams.append(new_ws)
        print(f"✅ Added '{name}'")
    
    # Show current state
    print("\nCurrent workstreams:")
    for i, ws in enumerate(project.workstreams, 1):
        print(f"   {i}. {ws.name} ({ws.duration_weeks}w)")

---

## 5️⃣ Dependency Analysis

### Analyze dependencies between workstreams

In [ ]:
# Initialize Dependency Agent
dependency_agent = DependencyAgent(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    verbose=True
)

print("✅ Dependency Agent initialized")

In [ ]:
# Analyze dependencies
print("\n🔗 Analyzing dependencies...\n")

dep_graph, reasoning = dependency_agent.analyze_dependencies(
    workstreams=project.workstreams,
    rag_engine=rag_engine,
    rfp_context=rfp_text[:2000]
)

# Save to project
project.dependencies = dep_graph
project.add_agent_reasoning(reasoning)

print("\n✅ Dependency analysis complete!")

In [ ]:
# Display dependency analysis
print("\n" + "="*70)
print("DEPENDENCY ANALYSIS RESULTS")
print("="*70)

# Map IDs to names for display
id_to_name = {ws.id: ws.name for ws in project.workstreams}

print("\n🔗 Dependencies:")
if dep_graph.workstream_dependencies:
    for ws_id, dep_ids in dep_graph.workstream_dependencies.items():
        if dep_ids:
            ws_name = id_to_name.get(ws_id, ws_id)
            dep_names = [id_to_name.get(d, d) for d in dep_ids]
            print(f"   • {ws_name}")
            print(f"     ↳ depends on: {', '.join(dep_names)}")
else:
    print("   No dependencies defined (all can start in parallel)")

print("\n⚡ Parallel Groups:")
if dep_graph.parallel_workstreams:
    for i, group in enumerate(dep_graph.parallel_workstreams, 1):
        group_names = [id_to_name.get(ws_id, ws_id) for ws_id in group]
        print(f"   {i}. {', '.join(group_names)}")
else:
    print("   No parallel groups identified")

print("\n🎯 Critical Path:")
if dep_graph.critical_path_workstreams:
    for i, ws_id in enumerate(dep_graph.critical_path_workstreams, 1):
        ws_name = id_to_name.get(ws_id, ws_id)
        print(f"   {i}. {ws_name}")
else:
    print("   Not determined yet")

### 🛠️ Human-in-the-Loop: Validate Dependencies

**Review the dependencies above. Make adjustments if needed.**

In [ ]:
# Manual dependency adjustment
print("\n🛠️ DEPENDENCY REFINEMENT")
print("="*70)
print("\nCurrent dependencies look correct?")
print("  Enter 'y' to continue")
print("  Enter 'n' to manually adjust")

response = input("\nYour choice (y/n): ").strip().lower()

if response == 'y':
    print("✅ Dependencies confirmed")
else:
    print("\n💡 Manual dependency editing coming in next version...")
    print("   For now, dependencies are as analyzed by AI")

---

## 6️⃣ Timeline Optimization

### Optimize the project timeline based on dependencies

In [ ]:
# Initialize Timeline Optimizer
optimizer = TimelineOptimizer(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    verbose=True
)

print("✅ Timeline Optimizer initialized")

In [ ]:
# Calculate sequential timeline (before optimization)
sequential_duration = sum(ws.duration_weeks for ws in project.workstreams)
print(f"📊 Sequential Timeline: {sequential_duration} weeks (~{sequential_duration/4:.1f} months)")

if project.total_duration_weeks:
    print(f"🎯 RFP Constraint: {project.total_duration_weeks} weeks")
    if sequential_duration > project.total_duration_weeks:
        print(f"⚠️ Exceeds constraint by {sequential_duration - project.total_duration_weeks} weeks")

In [ ]:
# Optimize timeline
print("\n⚡ Optimizing timeline...\n")

constraints = {}
if project.total_duration_weeks:
    constraints['max_duration_weeks'] = project.total_duration_weeks

opt_result, optimized_workstreams, reasoning = optimizer.optimize(
    workstreams=project.workstreams,
    dependencies=project.dependencies,
    rag_engine=rag_engine,
    constraints=constraints
)

# Save results
project.optimization_result = opt_result
project.optimized_duration_weeks = opt_result.optimized_duration_weeks
project.workstreams = optimized_workstreams  # Use optimized workstreams
project.add_agent_reasoning(reasoning)

print("\n✅ Timeline optimization complete!")

In [ ]:
# Display optimization results
print("\n" + "="*70)
print("TIMELINE OPTIMIZATION RESULTS")
print("="*70)

print(f"\n📊 Duration:")
print(f"   Original (sequential): {opt_result.original_duration_weeks} weeks")
print(f"   Optimized (with parallel): {opt_result.optimized_duration_weeks} weeks")
print(f"   ✅ Saved: {opt_result.weeks_saved} weeks")

print(f"\n⚡ Parallel Blocks ({len(opt_result.parallel_blocks)}):")
for i, block in enumerate(opt_result.parallel_blocks, 1):
    ws_names = [id_to_name.get(ws_id, ws_id) for ws_id in block.get('workstream_ids', [])]
    print(f"   {i}. Week {block.get('start_week', '?')}: {', '.join(ws_names)}")
    print(f"      Rationale: {block.get('rationale', 'N/A')[:80]}...")

if opt_result.risks:
    print(f"\n⚠️ Optimization Risks ({len(opt_result.risks)}):")
    for i, risk in enumerate(opt_result.risks, 1):
        print(f"   {i}. {risk.get('risk', 'Unknown')}")
        print(f"      Mitigation: {risk.get('mitigation', 'N/A')}")

if opt_result.recommendations:
    print(f"\n💡 Recommendations:")
    for i, rec in enumerate(opt_result.recommendations, 1):
        print(f"   {i}. {rec}")

### 🛠️ Human-in-the-Loop: Review Optimization

**Review the optimized timeline. Acceptable?**

In [ ]:
# Optimization review
print("\n🛠️ OPTIMIZATION REVIEW")
print("="*70)
print(f"\nOptimized timeline: {opt_result.optimized_duration_weeks} weeks")
print(f"Weeks saved: {opt_result.weeks_saved}")
print(f"\nAccept this optimization? (y/n): ")

response = input().strip().lower()

if response == 'y':
    print("\n✅ Optimization accepted!")
    print(f"   Final timeline: {opt_result.optimized_duration_weeks} weeks")
else:
    print("\n❌ Optimization rejected. Reverting to sequential timeline.")
    project.optimized_duration_weeks = None
    # Reset workstream start weeks to sequential
    current_week = 1
    for ws in project.workstreams:
        ws.start_week = current_week
        current_week += ws.duration_weeks

---

## 7️⃣ Detailed Breakdown: Modules & Activities

### Build detailed modules for each workstream

In [ ]:
# Initialize Module Agent
module_agent = ModuleAgent(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    verbose=True
)

print("✅ Module Agent initialized")

In [ ]:
# Build modules for each workstream
print("\n🏗️ Building detailed modules for all workstreams...\n")

for i, workstream in enumerate(project.workstreams, 1):
    print(f"\n{'='*70}")
    print(f"Building modules for Workstream {i}/{len(project.workstreams)}")
    print(f"{'='*70}")
    
    modules, reasoning = module_agent.build_modules(
        workstream=workstream,
        all_workstreams=project.workstreams,
        rag_engine=rag_engine,
        project_context=rfp_text[:2000]
    )
    
    # Assign modules to workstream
    workstream.modules = modules
    
    # Save reasoning
    project.add_agent_reasoning(reasoning)
    
    print(f"\n✅ Generated {len(modules)} modules for '{workstream.name}'")

print("\n" + "="*70)
print("✅ ALL MODULES BUILT")
print("="*70)

total_modules = sum(len(ws.modules) for ws in project.workstreams)
total_activities = sum(
    len(mod.activities)
    for ws in project.workstreams
    for mod in ws.modules
)

print(f"\n📊 Summary:")
print(f"   Workstreams: {len(project.workstreams)}")
print(f"   Total modules: {total_modules}")
print(f"   Total activities: {total_activities}")

In [ ]:
# Validate for duplicate module names
validation_issues = project.validate_structure()

if validation_issues:
    print("\n⚠️ VALIDATION ISSUES:")
    for issue in validation_issues:
        print(f"   • {issue}")
else:
    print("\n✅ Structure validation passed! No issues found.")

In [ ]:
# Display complete structure
print("\n" + "="*70)
print("COMPLETE PROJECT STRUCTURE")
print("="*70)

for ws_idx, ws in enumerate(project.workstreams, 1):
    print(f"\n📦 WORKSTREAM {ws_idx}: {ws.name}")
    print(f"   Duration: {ws.duration_weeks} weeks (Week {ws.start_week}-{ws.end_week})")
    print(f"   Modules: {len(ws.modules)}")
    
    for mod_idx, module in enumerate(ws.modules, 1):
        print(f"\n   📋 Module {ws_idx}.{mod_idx}: {module.name}")
        print(f"      Duration: {module.duration_weeks} weeks (Week {module.start_week}-{module.end_week})")
        print(f"      Deliverables: {len(module.deliverables)}")
        if module.deliverables:
            for deliv in module.deliverables[:2]:
                print(f"         • {deliv}")
        print(f"      Activities: {len(module.activities)}")
        
        if module.parallel_with:
            print(f"      ⚡ Parallel with: {', '.join(module.parallel_with)}")

---

## 8️⃣ Save Progress

### Save your project to disk

In [ ]:
# Save project
saved_path = save_project(project)
print(f"\n💾 Project saved to: {saved_path}")

# Print summary
print_project_summary(project)

---

## 9️⃣ Next Steps

### What to do next:

1. **Review Agent Reasoning** - See how AI made decisions
2. **Generate Visualizations** - Create professional timeline charts
3. **Export to Excel/PowerPoint** - Generate client-ready deliverables

Continue to the next notebooks:
- `03_Visualization_Studio.ipynb` - Create custom charts and timelines
- `04_Export_Templates.ipynb` - Generate professional exports

---

## 🔍 Agent Reasoning Review

### View detailed reasoning from all AI agents

In [ ]:
# Display all agent reasoning
print(f"\n📚 Total agent operations: {len(project.agent_reasoning_log)}\n")

for i, reasoning in enumerate(project.agent_reasoning_log, 1):
    print(f"\n{'='*70}")
    print(f"AGENT OPERATION {i}/{len(project.agent_reasoning_log)}")
    print(format_reasoning_for_display(reasoning))

---

## 📊 Quick Visualization (Basic)

### Simple timeline visualization

In [ ]:
# Simple ASCII timeline
print("\n" + "="*70)
print("PROJECT TIMELINE (Simple View)")
print("="*70 + "\n")

total_weeks = max(ws.end_week for ws in project.workstreams)

for ws in project.workstreams:
    # Create timeline bar
    bar = [" "] * total_weeks
    for week in range(ws.start_week - 1, ws.end_week):
        bar[week] = "█"
    
    timeline_str = "".join(bar)
    
    print(f"{ws.name[:40]:40} | {timeline_str} | W{ws.start_week}-{ws.end_week}")

print("\n" + "-"*70)
print(f"Total Duration: {total_weeks} weeks (~{total_weeks/4:.1f} months)")
print("="*70)

---

## ✅ Completion Checklist

- ✅ RFP analyzed and workstreams extracted
- ✅ Dependencies analyzed
- ✅ Timeline optimized
- ✅ Detailed modules built
- ✅ Project saved

**Next:** Continue to visualization and export notebooks!

---

*Generated by AI-Powered Enterprise RAG Architecture v2.0*